# 실습 5. 비율을 비교하는 방법

사회약학 연구방법 노트의 '실습 5. 비율을 비교하는 방법'에 딸린 노트북입니다. 앞부분(a–c)은 MASS 패키지의 birthwt 자료(출생아와 산모 189명)로 흡연, 고혈압 병력, 인종과 저체중아 출생의 관련을 카이제곱 검정과 Fisher의 정확한 검정으로 비교하고 위험차, 상대위험도, 오즈비를 구합니다. 뒷부분(d)은 식도암 환자-대조군 자료 esoph로 음주량에 따른 추세를 검정합니다. 셀을 위에서부터 차례로 실행하세요. 설명과 출력 읽는 법은 사이트의 실습 5에 있습니다.


## a. 실습 데이터 준비

이 실습이 쓰는 패키지를 불러와 버전을 확인한 뒤 birthwt 자료를 읽습니다. birthwt와 d 절의 esoph는 인터넷에서 읽어 오므로 인터넷 연결이 필요합니다.

### 셀 1. 패키지 불러오기와 버전 확인

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import scipy, statsmodels, matplotlib

print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("scipy       ", scipy.__version__)
print("statsmodels ", statsmodels.__version__)
print("matplotlib  ", matplotlib.__version__)

### 셀 2. 데이터 불러오기

In [ ]:
url = ("https://vincentarelbundock.github.io/Rdatasets/"
       "csv/MASS/birthwt.csv")
df = pd.read_csv(url)
print(df.shape)
df.head()

### 셀 3. 범주에 이름표와 순서 붙이기

In [ ]:
def label(col, mapping):
    # mapping에 적은 순서가 곧 범주의 순서가 됩니다
    return pd.Categorical(df[col].map(mapping),
                          categories=list(mapping.values()))

df["ptl_any"] = (df["ptl"] > 0).astype(int)  # 조산 경험 1회 이상
df["low_f"] = label("low", {1: "LBW", 0: "Not LBW"})
df["smoke_f"] = label("smoke", {1: "Smoker", 0: "Nonsmoker"})
df["ht_f"] = label("ht", {1: "HT", 0: "No HT"})
df["ptl_f"] = label("ptl_any", {1: "Preterm", 0: "None"})
df["race_f"] = label("race", {1: "White", 2: "Black", 3: "Other"})

### 셀 4. 범주별 빈도 확인

In [ ]:
for col in ["low_f", "smoke_f", "ht_f", "ptl_f", "race_f"]:
    print(col, df[col].value_counts(sort=False).to_dict())
print("LBW 비율:", round(df["low"].mean(), 3))

## b. 카이제곱 검정

### 셀 5. 교차표와 합계

In [ ]:
pd.crosstab(df["smoke_f"], df["low_f"], margins=True)

### 셀 6. 행 백분율

In [ ]:
pct = pd.crosstab(df["smoke_f"], df["low_f"], normalize="index")
(pct * 100).round(1)

### 셀 7. 카이제곱 검정 (연속성 보정 없음)

In [ ]:
obs = pd.crosstab(df["smoke_f"], df["low_f"])  # 합계 없는 2×2 표
res = stats.chi2_contingency(obs, correction=False)
print("statistic    :", res.statistic)
print("pvalue       :", res.pvalue)
print("dof          :", res.dof)
print("expected_freq:")
print(res.expected_freq)

### 셀 8. Yates 연속성 보정과 비교

In [ ]:
yates = stats.chi2_contingency(obs)  # correction=True가 기본값
print(f"Pearson: chi2 = {res.statistic:.3f}, p = {res.pvalue:.4f}")
print(f"Yates:   chi2 = {yates.statistic:.3f}, p = {yates.pvalue:.4f}")
print("가장 작은 기대빈도:", res.expected_freq.min().round(1))

### 셀 9. 상대위험도와 오즈비

In [ ]:
from statsmodels.stats.contingency_tables import Table2x2

t22 = Table2x2(obs.to_numpy())  # DataFrame 대신 숫자 배열로
print(t22.summary())

### 셀 10. DataFrame을 그대로 넣으면

In [ ]:
wrong = Table2x2(obs)  # 이름표가 붙은 DataFrame을 그대로 넣으면
print(wrong.table.shape, wrong.riskratio)

### 셀 11. 위험차와 95% 신뢰구간

In [ ]:
from statsmodels.stats.proportion import confint_proportions_2indep

(a, b), (c, d) = obs.to_numpy()     # 네 칸의 빈도
p1, p0 = a / (a + b), c / (c + d)   # 흡연군, 비흡연군의 위험
lo, hi = confint_proportions_2indep(a, a + b, c, c + d,
                                    compare="diff", method="wald")
print(f"p1 = {p1:.3f}, p0 = {p0:.3f}")
print(f"RD = {p1 - p0:.3f} (95% CI {lo:.3f} to {hi:.3f})")

### 셀 12. 오즈비와 상대위험도의 관계

In [ ]:
rr, or_ = t22.riskratio, t22.oddsratio
print(f"RR = {rr:.3f}, OR = {or_:.3f}")
print("RR x (1-p0)/(1-p1) =", round(rr * (1 - p0) / (1 - p1), 3))
print("OR/(1-p0+p0*OR) =", round(or_ / (1 - p0 + p0 * or_), 3))

### 셀 13. 세 군 이상: 인종별 저체중아 비율

In [ ]:
obs_r = pd.crosstab(df["race_f"], df["low_f"])
res_r = stats.chi2_contingency(obs_r)  # 2×2가 아니면 보정 없음
print(f"chi2 = {res_r.statistic:.3f}, dof = {res_r.dof}, "
      f"p = {res_r.pvalue:.4f}")
pct_r = pd.crosstab(df["race_f"], df["low_f"], normalize="index")
(pct_r * 100).round(1)

### 셀 14. 수정 표준화 잔차

In [ ]:
from statsmodels.stats.contingency_tables import Table

Table(obs_r).standardized_resids.round(2)

### 셀 15. 군별 저체중아 비율 그래프

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(8, 3.2), sharey=True)
for ax, col in zip(axes, ["smoke_f", "race_f"]):
    pct = df.groupby(col, observed=True)["low"].mean() * 100
    bars = ax.bar(pct.index.astype(str), pct.values)
    ax.bar_label(bars, fmt="%.1f")
    ax.set_xlabel(col)
axes[0].set_ylabel("Low birth weight (%)")
axes[0].set_ylim(0, 50)
plt.show()

## c. Fisher의 정확한 검정

### 셀 16. 고혈압 병력과 저체중아: 기대빈도 확인

In [ ]:
obs_h = pd.crosstab(df["ht_f"], df["low_f"])
print(obs_h)
exp_h = stats.chi2_contingency(obs_h).expected_freq
print(exp_h.round(2))
print("기대빈도 5 미만인 칸:", (exp_h < 5).sum(), "/ 4")

### 셀 17. Fisher의 정확한 검정

In [ ]:
fe = stats.fisher_exact(obs_h)
print("statistic:", fe.statistic)
print("pvalue   :", fe.pvalue)
chi_u = stats.chi2_contingency(obs_h, correction=False)
chi_y = stats.chi2_contingency(obs_h)
print(f"Pearson p = {chi_u.pvalue:.4f}, Yates p = {chi_y.pvalue:.4f}")

### 셀 18. 오즈비 두 가지

In [ ]:
from scipy.stats.contingency import odds_ratio

for kind in ["sample", "conditional"]:
    o = odds_ratio(obs_h.to_numpy(), kind=kind)
    ci = o.confidence_interval(confidence_level=0.95)
    print(f"{kind:11s} OR = {o.statistic:.3f} "
          f"(95% CI {ci.low:.2f} to {ci.high:.2f})")

### 셀 19. 가능한 모든 표의 확률

In [ ]:
total, lbw, ht_n = 189, 59, 12   # 전체, LBW 합계, HT 산모 수
a = np.arange(0, ht_n + 1)        # HT 산모 중 LBW 수: 0 ~ 12
prob = stats.hypergeom.pmf(a, total, lbw, ht_n)
p_obs = prob[7]                   # 관측된 표 (a = 7)
extreme = prob <= p_obs * (1 + 1e-7)
print(prob.round(4).tolist())
print(f"P(a = 7) = {p_obs:.4f}, 양측 p = {prob[extreme].sum():.4f}")

### 셀 20. 초기하분포 그래프

In [ ]:
plt.figure(figsize=(6, 3))
plt.bar(a, prob, color=np.where(extreme, "tab:orange", "tab:gray"))
plt.axvline(ht_n * lbw / total, ls="--", color="k")  # 기대 3.75
plt.xlabel("Number of LBW among 12 mothers with hypertension")
plt.ylabel("Probability")
plt.show()

## d. 선형 대 선형 결합

### 셀 21. esoph 데이터 불러오기

In [ ]:
url2 = ("https://vincentarelbundock.github.io/Rdatasets/"
        "csv/datasets/esoph.csv")
es = pd.read_csv(url2)
print(es.shape, es["ncases"].sum(), es["ncontrols"].sum())
es.head(6)

### 셀 22. 순서를 지정하지 않고 합치면

In [ ]:
es.groupby("alcgp")[["ncases", "ncontrols"]].sum()

### 셀 23. 음주량 순서대로 합치기

In [ ]:
alc_order = ["0-39g/day", "40-79", "80-119", "120+"]
es["alcgp"] = pd.Categorical(es["alcgp"], categories=alc_order,
                             ordered=True)
alc = (es.groupby("alcgp", observed=True)
         [["ncases", "ncontrols"]].sum())
n_grp = alc["ncases"] + alc["ncontrols"]
alc["pct_case"] = 100 * alc["ncases"] / n_grp
alc["odds"] = alc["ncases"] / alc["ncontrols"]
alc["OR"] = alc["odds"] / alc["odds"].iloc[0]
alc.round(3)

### 셀 24. 음주량별 오즈비와 95% 신뢰구간

In [ ]:
ref = alc.iloc[0]            # 기준범주: 0-39 g/day
for g, row in alc.iloc[1:].iterrows():
    t = Table2x2(np.array([[row["ncases"], row["ncontrols"]],
                           [ref["ncases"], ref["ncontrols"]]]))
    lo, hi = t.oddsratio_confint()
    print(f"{g:>6}: OR {t.oddsratio:5.2f} ({lo:.2f} to {hi:.2f})")

### 셀 25. 환자군과 대조군의 음주 분포

In [ ]:
cc = alc[["ncases", "ncontrols"]]   # 환자 수, 대조군 수 두 열만
(cc / cc.sum() * 100).round(1)       # 열 백분율

### 셀 26. 음주량별 환자 비율 그래프

In [ ]:
plt.figure(figsize=(6, 3))
bars = plt.bar(alc.index.astype(str), alc["pct_case"])
plt.bar_label(bars, labels=[f"{x}/{m}" for x, m
                            in zip(alc["ncases"], n_grp)])
plt.xlabel("Alcohol consumption (g/day)")
plt.ylabel("Cases among subjects (%)")
plt.ylim(0, 80)
plt.show()

### 셀 27. Pearson 카이제곱 검정 (자유도 3)

In [ ]:
pear = stats.chi2_contingency(cc)
print(f"Pearson chi2 = {pear.statistic:.2f}, dof = {pear.dof}, "
      f"p = {pear.pvalue:.2e}")

### 셀 28. M² = (N − 1)r² 직접 계산

In [ ]:
score = np.array([1, 2, 3, 4])     # 음주 범주의 점수
case_n = cc["ncases"].to_numpy()
ctrl_n = cc["ncontrols"].to_numpy()
x = np.r_[np.repeat(score, case_n), np.repeat(score, ctrl_n)]
y = np.r_[np.ones(case_n.sum()), np.zeros(ctrl_n.sum())]
print(x.shape, x[y == 1].mean(), x[y == 0].mean().round(3))
r = np.corrcoef(x, y)[0, 1]
M2 = (len(x) - 1) * r**2
print(f"r = {r:.4f}, M2 = {M2:.2f}, p = {stats.chi2.sf(M2, 1):.2e}")

### 셀 29. statsmodels로 같은 검정

In [ ]:
lbl = Table(cc).test_ordinal_association(
    row_scores=score, col_scores=np.array([1, 0]))
print(f"z = {lbl.zscore:.3f}, z^2 = {lbl.zscore**2:.2f}, "
      f"p = {lbl.pvalue:.2e}")
print(f"Cochran-Armitage (N r^2) = {len(x) * r**2:.2f}")
dev = pear.statistic - M2          # 직선에서 벗어난 부분
print(f"departure = {dev:.2f}, p = {stats.chi2.sf(dev, 2):.3f}")

## 과제 정답

**과제 1.** 셀 24의 출력에서 80–119 g 행(OR 7.80, 95% CI 4.68–13.02)을 한 문장으로 옮기고, 이 숫자가 말해 주지 않는 것을 두 가지 이상 쓰세요. 대조군을 두 배로 뽑았다면 이 군의 환자 비율과 오즈비가 어떻게 되는지, 40–79 g군과 견준 오즈비는 얼마인지도 계산해 보세요.

### 셀 30. 과제 1 정답. 음주량 80–119 g군의 오즈비 읽기

In [ ]:
row, ref = alc.loc["80-119"], alc.iloc[0]   # 80-119 g군, 기준군(0-39 g)
t80 = Table2x2(np.array([[row["ncases"], row["ncontrols"]],
                         [ref["ncases"], ref["ncontrols"]]]))
lo, hi = t80.oddsratio_confint()
print(f"80-119 vs 0-39: OR {t80.oddsratio:.2f} ({lo:.2f} to {hi:.2f})")
for k in [1, 2]:                            # 대조군을 k배로 뽑았다면
    pct80 = 100 * row["ncases"] / (row["ncases"] + k * row["ncontrols"])
    pct0 = 100 * ref["ncases"] / (ref["ncases"] + k * ref["ncontrols"])
    or_k = ((row["ncases"] / (k * row["ncontrols"]))
            / (ref["ncases"] / (k * ref["ncontrols"])))
    print(f"controls x{k}: case % {pct80:.1f} vs {pct0:.1f}, OR {or_k:.2f}")
mid = alc.loc["40-79"]                      # 이웃한 40-79 g군
print("80-119 vs 40-79: OR", round((row["ncases"] / row["ncontrols"])
                                   / (mid["ncases"] / mid["ncontrols"]), 2))

**과제 2.** 셀 9의 `Table2x2`는 첫 행을 노출군, 첫 열을 사건으로 읽습니다. 흡연 교차표 `obs`의 행 순서를 바꿔 비흡연군을 첫 행에 두거나, 열 순서를 바꿔 'Not LBW'를 사건으로 두고 위험차, 상대위험도, 오즈비를 다시 구하세요. 각 지표가 어떻게 바뀌는지, 열을 바꿨을 때 상대위험도가 원래 값의 역수(1 ÷ 1.608 = 0.622)가 되지 않는 이유는 무엇인지 설명하세요.

### 셀 31. 과제 2 정답. 행과 열의 순서 바꾸기

In [ ]:
tabs = {"original": obs.to_numpy(),                 # 흡연군, LBW가 첫 행·첫 열
        "rows swapped": obs.to_numpy()[::-1],        # 비흡연군을 첫 행으로
        "columns swapped": obs.to_numpy()[:, ::-1]}  # 'Not LBW'를 첫 열(사건)로
for name, arr in tabs.items():
    t = Table2x2(arr)
    r1, r0 = arr[0, 0] / arr[0].sum(), arr[1, 0] / arr[1].sum()
    print(f"{name:15s} risks {r1:.3f} {r0:.3f}  RD {r1 - r0:+.3f}"
          f"  RR {t.riskratio:.3f}  OR {t.oddsratio:.3f}")

**과제 3.** 동료가 논문의 표에 있는 비율을 그대로 쓰려고 고혈압 병력과 저체중아의 행 백분율 표를 만들어 `stats.chi2_contingency(pct_h, correction=False)`에 넣었습니다(`pct_h = pd.crosstab(df["ht_f"], df["low_f"], normalize="index") * 100`). 그리고 "고혈압 병력이 있는 산모는 저체중아 출산 위험이 3.37배 높았다(χ² = 17.02, P < 0.001)"라고 썼습니다. 동료의 계산을 재현하고, 이 결론에서 틀린 곳 두 가지를 찾아 왜 틀렸는지 설명하세요.

### 셀 32. 과제 3 정답. 백분율 표로 한 카이제곱 검정

In [ ]:
pct_h = pd.crosstab(df["ht_f"], df["low_f"], normalize="index") * 100
print(pct_h.round(1))
bad = stats.chi2_contingency(pct_h, correction=False)   # 동료의 계산
print(f"percent table: total {pct_h.to_numpy().sum():.0f}, "
      f"chi2 = {bad.statistic:.2f}, p = {bad.pvalue:.1e}")
print(f"OR from percents {Table2x2(pct_h.to_numpy()).oddsratio:.3f}, "
      f"from counts {Table2x2(obs_h.to_numpy()).oddsratio:.3f}")
rr_h = Table2x2(obs_h.to_numpy()).riskratio             # 실제 인원으로
print(f"counts: Fisher p = {fe.pvalue:.3f}, RR = {rr_h:.2f}")